# 02 · Tools

In notebook 01 the model invented ShipFast's refund policy and an order status, because it cannot see
either. **Tools** fix that: functions the model can ask your code to run, whose results it then reads.

**You will learn**

- where the tools live, and why they are plain Python functions
- how LangChain turns a function into a tool the model can see
- what a tool call is, and how to run one and hand the result back
- how tool errors and safety checks work when the model gets something wrong

In [1]:
# Check that Ollama is running and the shared model exists, then build the shared chat model (lg_agents/llm.py).
from agent_contracts.harness import verify_model
from lg_agents.llm import make_llm

verify_model()
llm = make_llm()  # the same model setup as notebook 01

## The tools are plain functions

ShipFast's five tools live once, in `agent_contracts.shipfast.tools`, shared by all three tutorials. They
know nothing about LangChain: each takes simple arguments and returns a JSON string. You can call them
directly.

In [2]:
# The shared tools live in contracts, not in LangChain. TOOLS is the list of the five plain functions.
from agent_contracts.shipfast import tools as shipfast

shipfast.reset()  # start clean: clears the refund ledger, the tickets and the tool-call log
print("ShipFast's tools:", ", ".join(f.__name__ for f in shipfast.TOOLS))

ShipFast's tools: lookup_order, track_shipment, search_policy, issue_refund, escalate_to_human


Call one directly, like any function. `lookup_order` takes an order ID and returns a JSON string, which
the cell below parses so it prints as one field per line.

In [3]:
import json

# Call a tool by hand: the argument is an order ID, the return value is a JSON string.
order = json.loads(shipfast.lookup_order("A1001"))

for field, value in order.items():
    print(f"{field:13}{value}")

order_id     A1001
customer     Maria Santos
item         Wireless Headphones
total        89.0
status       in_transit
carrier      LBC
ordered_on   2026-10-28
eta          2026-11-04
delivered_on None


## Turning a function into a tool

LangChain's `tool()` reads a function's name, type hints and docstring, and builds the description the
model sees: a name, what it does, and a JSON schema for its arguments. `lg_agents/tools.py` wraps all
five; this is all of it:

In [4]:
# shipfast_tools() lives in lg_agents/tools.py. Its source is shown because it is the whole wrapper:
# LangChain's tool() turns each plain function into a tool.
import inspect

from lg_agents.tools import shipfast_tools

print(inspect.getsource(shipfast_tools))

def shipfast_tools() -> list[BaseTool]:
    return [tool(f) for f in shipfast.TOOLS]



Build the tools and look at what the model is told about each one: its name and its arguments, with their
types.

In [5]:
tools = shipfast_tools()  # one LangChain tool per function

print(f"{'tool':18} arguments")
for t in tools:
    # t.args is the argument schema LangChain built from the function's type hints
    arguments = ", ".join(f"{name}: {spec['type']}" for name, spec in t.args.items())
    print(f"{t.name:18} {arguments}")

tool               arguments
lookup_order       order_id: string
track_shipment     order_id: string
search_policy      query: string
issue_refund       order_id: string, amount: number, reason: string
escalate_to_human  order_id: string, summary: string


The docstring is the only instruction the model gets about when to use a tool, so it matters as much
as the code. `issue_refund`'s docstring, for example, says the amount cannot exceed the order total.

## A tool call

`bind_tools` attaches the tool descriptions to every request. The model can then answer in text, or
reply with a **tool call**: the name of a tool and the arguments to run it with. It does not run anything
itself.

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage

# The instructions for this notebook: check facts with the tools instead of guessing.
SPECIALIST = ("You are ShipFast's support specialist. Use the tools to check facts before you answer; "
              "never guess order details or policy.")

# bind_tools attaches the five tool descriptions to every request this model object sends.
llm_with_tools = llm.bind_tools(tools)

Now send the customer's message and look at the reply. When the model wants a tool, the text is empty and
the request is in `tool_calls`.

In [7]:
messages = [SystemMessage(SPECIALIST), HumanMessage("Hi, where is my order A1001?")]
ai = llm_with_tools.invoke(messages)  # an AIMessage: either text, or a request to run a tool

print("text:      ", repr(ai.content))  # empty when the model asks for a tool instead of answering
print("tool calls:", len(ai.tool_calls))
for call in ai.tool_calls:
    print("  tool:     ", call["name"])
    print("  arguments:", call["args"])

text:       ''
tool calls: 1
  tool:      track_shipment
  arguments: {'order_id': 'A1001'}


## Running the call and handing back the result

Your code runs the tool. Invoking a LangChain tool with the tool call itself returns a `ToolMessage`
that carries the call's id, so the model can match result to request. Append both messages and ask
again: now the answer is grounded in real data.

In [8]:
# Look tools up by name, so a tool call from the model can be matched to the tool to run.
tools_by_name = {t.name: t for t in tools}

messages.append(ai)  # the conversation needs the model's own turn (its tool call) before the result
for call in ai.tool_calls:
    # Invoking a tool with the whole tool call returns a ToolMessage that carries the call's id.
    result = tools_by_name[call["name"]].invoke(call)
    print("tool ran:    ", call["name"])
    print("result type: ", type(result).__name__)
    print("result start:", result.content[:120], "...")  # the start of the JSON the tool returned
    messages.append(result)

tool ran:     track_shipment
result type:  ToolMessage
result start: {"order_id": "A1001", "status": "in_transit", "carrier": "LBC", "eta": "2026-11-04", "delivered_on": null, "tracking": [ ...


With the result in the conversation, ask the model again. This time it has real data to answer from.

In [9]:
final = llm_with_tools.invoke(messages)

print("tool calls:", len(final.tool_calls))
print("answer:    ", final.content)

tool calls: 0
answer:     Your order A1001 is currently in transit with LBC. Here's the latest update:

- **Status:** In Transit
- **Carrier:** LBC
- **Estimated Time of Arrival (ETA):** November 4, 2026

**Tracking Events:**
- October 28, 2026: Order packed at Manila warehouse
- October 30, 2026: Picked up by LBC
- October 31, 2026: Arrived at Cebu sorting hub

If you have any further questions, feel free to ask!


That is the whole idea of an agent, done by hand: call the model, run the tools it asks for, feed the
results back, repeat until it answers in text. Notebook 03 lets LangGraph run that loop.

## Which tool does it reach for?

Here is the model's **first** move on the opening message of each ShipFast scenario. Nothing is run; we
only look at what it asks for.

In [10]:
# Show the model's first move for the opening message of every ShipFast scenario. Nothing is run.
from agent_contracts.shipfast.scoring import load_scenarios

seen = set()  # opening messages already tried: two scenarios start with the same one
with_tool = 0  # how many messages started with a tool call
print(f"{'customer message':50} -> first move")
for scenario in load_scenarios():
    text = scenario["turns"][0]  # the customer's first message
    if text in seen:
        continue
    seen.add(text)
    first = llm_with_tools.invoke([SystemMessage(SPECIALIST), HumanMessage(text)])
    if first.tool_calls:
        # each call as name(arg=value, ...)
        asked = ", ".join(f"{c['name']}({', '.join(f'{k}={v!r}' for k, v in c['args'].items())})"
                          for c in first.tool_calls)
        with_tool += 1
    else:
        asked = "(no tool) " + first.content[:60]
    print(f"{text[:48]:50} -> {asked}")  # long messages are cut to 48 characters

print(f"\n{with_tool} of {len(seen)} messages started with a tool call")

customer message                                   -> first move


Hi, where is my order A1001?                       -> track_shipment(order_id='A1001')


My order A1002 never arrived and tracking says i   -> track_shipment(order_id='A1002')


Order A1003 is more than a week late. Please can   -> track_shipment(order_id='A1003')


My vase set from order A1004 arrived broken.       -> track_shipment(order_id='A1004')


I want to return order A1005, the stand wobbles.   -> lookup_order(order_id='A1005')


Where is order A9999?                              -> track_shipment(order_id='A9999')


Can you write me a poem about cats?                -> (no tool) I can't write poems, but I can help you find information or 

6 of 7 messages started with a tool call


Look for two things: whether each choice makes sense for the message, and the out-of-scope request,
where the right move is no tool at all.

When we ran it, the model opened every in-scope message with `track_shipment`, refunds included, except the wobbly stand
(A1005), where it started with `lookup_order`. Opening with a lookup or a tracking check is
a reasonable first step, but a refund also needs the policy and the order total, so one tool call is not
enough. Getting from the first call to a correct decision takes several rounds, which is exactly the loop
notebook 03 automates.

## When a tool says no

Tools report problems as data, not exceptions, so the model can read them and recover. An unknown order
comes back as `order_not_found`, and policy P6 says to ask the customer to check the ID.

In [11]:
# The by-hand loop again, on an order that does not exist. A few rounds: ask, run the tools, ask again.
messages = [SystemMessage(SPECIALIST), HumanMessage("Where is order A9999?")]
for _ in range(3):
    ai = llm_with_tools.invoke(messages)
    messages.append(ai)
    if not ai.tool_calls:
        break  # the model answered in text, so the loop is done
    for call in ai.tool_calls:
        result = tools_by_name[call["name"]].invoke(call)
        print("tool called: ", call["name"], call["args"])
        print("tool result: ", result.content)
        messages.append(result)
print("final answer:", ai.content)

tool called:  track_shipment {'order_id': 'A9999'}
tool result:  {"error": "order_not_found", "order_id": "A9999"}


final answer: The order ID "A9999" could not be found. Please double-check the order ID and try again. If you need further assistance, feel free to ask!


## Safety lives in the tool, not the prompt

In one of our early test runs, the model once tried to refund $120 on a $45 order without looking up
the total. The prompt cannot prevent that; the tool can. `issue_refund` refuses any amount above the
order total, whoever calls it.

In [12]:
# Try to refund $120 on an order that cost $45 (A1002), then the correct $45. The tool judges both itself.
too_much = shipfast.issue_refund("A1002", 120, "lost package")
correct = shipfast.issue_refund("A1002", 45, "lost package")

print("refund of $120:", too_much)
print("refund of $45: ", correct)

refund of $120: {"error": "amount_exceeds_order_total", "order_id": "A1002", "amount": 120.0, "order_total": 45.0}
refund of $45:  {"refund_id": "RF-A1002", "order_id": "A1002", "amount": 45.0, "status": "issued"}


Only one of those should have reached the ledger, the record of refunds that were really issued.

In [13]:
# refunds() is the ledger: one entry per refund that was actually issued.
print("ledger:")
for r in shipfast.refunds():
    print(f"  {r['refund_id']}  order {r['order_id']}  ${r['amount']:.2f}  {r['status']}  ({r['reason']})")

ledger:
  RF-A1002  order A1002  $45.00  issued  (lost package)


The ledger is what ShipFast's test runner trusts: a result only counts as `refunded` if the ledger
has a matching refund. The model's claim is never enough.

The shared tools also record every call, whichever framework made it. Notebook 07 uses this so all three
tutorials count tool calls the same way.

In [14]:
# Every call to a shared tool is logged, whichever framework made it. Read the log, then reset it.
print("tools called, in order:", ", ".join(shipfast.tool_calls()))
shipfast.reset()  # leave a clean state behind

tools called, in order: lookup_order, track_shipment, track_shipment, issue_refund, issue_refund


## How the others do this

- **CrewAI** wraps the same functions with its own `@tool` decorator and gives them to an `Agent`; the
  agent runs them itself, so you rarely see a raw tool call.
- **Strands** also uses a `@tool` decorator, and its `Agent` runs the call-execute-repeat loop for you
  from the first example.

All three wrap the same five functions from `agent_contracts`, so the tools behave identically; only the
wrapping differs.

## Takeaways

- A tool is a plain function plus a description the model reads; keep the logic outside the framework.
- The model only *asks* for a tool. Your code runs it and hands back the result.
- Errors come back as data the model can act on, and safety checks belong in the tool.
- Next: **03 · Agent**, where LangGraph runs the loop for you.